# Crop Cnn Stability Experiments

Ce notebook reprend le script `crop_cnn_stability_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Stabilite des modeles crop utilisables dans une politique live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated split validation for crop CNN attention/PPE models.
- Commande de reproduction referencee : crop CNN stability.
- Artefacts controles : Crop CNN repeated split stability exists. (`runs/exp_016_crop_cnn_stability/metrics/crop_stability_summary.csv`); 20-seed crop CNN repeated split stability exists. (`runs/exp_045_crop_cnn_stability_20seed/metrics/crop_stability_summary.csv`).
- Run par defaut : `runs/exp_016_crop_cnn_stability`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "crop_cnn_stability_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from crop_cnn_experiments import CropDataset, TARGETS, predict, train_crop_model, video_metrics
from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, make_run_dir


## Fonction `parent_combo_split`

Cette cellule definit `parent_combo_split`. Elle prepare une partie du script.

In [ ]:
def parent_combo_split(index, seed):
    videos = (
        index.groupby("video_id", as_index=False)
        .agg(attention_label=("attention_label", "max"), blouse_label=("blouse_label", "max"))
        .copy()
    )
    videos["combo"] = videos["attention_label"].astype(str) + "_" + videos["blouse_label"].astype(str)
    labels = videos["combo"].to_numpy()
    if videos["combo"].value_counts().min() < 3:
        labels = videos["attention_label"].astype(str).to_numpy()
    train_ids, temp_ids, _, temp_labels = train_test_split(
        videos["video_id"].to_numpy(),
        labels,
        test_size=0.30,
        random_state=seed,
        stratify=labels,
    )
    temp_counts = pd.Series(temp_labels).value_counts()
    stratify_temp = temp_labels if temp_counts.min() >= 2 else None
    val_ids, test_ids = train_test_split(
        temp_ids,
        test_size=0.50,
        random_state=seed + 1,
        stratify=stratify_temp,
    )
    split = {video_id: "train" for video_id in train_ids}
    split.update({video_id: "val" for video_id in val_ids})
    split.update({video_id: "test" for video_id in test_ids})
    return split


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics):
    rows = []
    for (target, arch, split), group in metrics.groupby(["target", "architecture", "split"]):
        rows.append(
            {
                "target": target,
                "architecture": arch,
                "split": split,
                "n_repeats": int(group["repeat_seed"].nunique()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "roc_auc_mean": float(group["roc_auc"].mean()),
                "f1_mean": float(group["f1"].mean()),
                "f1_std": float(group["f1"].std(ddof=0)),
                "balanced_accuracy_mean": float(group["balanced_accuracy"].mean()),
                "balanced_accuracy_std": float(group["balanced_accuracy"].std(ddof=0)),
            }
        )
    return pd.DataFrame(rows)


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source_run = Path(args.crop_run)
    if not source_run.is_absolute():
        source_run = ROOT / source_run
    run_dir = make_run_dir(args.run_name)
    source_index = pd.read_csv(source_run / "features" / "crop_cnn_index.csv")
    index = source_index.copy()
    # Reuse crop images by pointing paths back to the source crop run.
    index["path"] = index["path"].apply(lambda p: str(source_run / p))
    write_json(
        run_dir / "config.json",
        {
            "crop_run": str(source_run),
            "seeds": args.seeds,
            "architectures": args.architectures,
            "epochs": args.epochs,
            "patience": args.patience,
            "split_policy": "parent video split stratified by attention/blouse combination",
        },
    )
    all_metrics = []
    all_history = []
    split_rows = []
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)

    # CropDataset joins run_dir/path. Absolute paths should survive Path joining on Windows.
    for seed in args.seeds:
        split = parent_combo_split(index, seed)
        split_index = index.copy()
        split_index["split"] = split_index["video_id"].map(split)
        split_index.to_csv(run_dir / "features" / f"crop_split_seed_{seed}.csv", index=False)
        video_split = split_index.groupby(["split", "video_id"]).size().reset_index()
        split_rows.append({"seed": seed, **video_split["split"].value_counts().to_dict()})
        for target in TARGETS:
            for arch in args.architectures:
                print(f"training seed{seed} {target} {arch}")
                model, history, train_time_s, model_size = train_crop_model(run_dir, split_index, target, arch, args, device)
                model_path = run_dir / "models" / f"{target}_{arch}.pt"
                renamed = run_dir / "models" / f"{target}_seed{seed}_{arch}.pt"
                if model_path.exists():
                    model_path.replace(renamed)
                for row in history:
                    row["repeat_seed"] = seed
                all_history.extend(history)
                eval_loader = DataLoader(
                    CropDataset(run_dir, split_index, target, train=False, image_size=args.image_size),
                    batch_size=args.batch_size,
                    shuffle=False,
                    num_workers=0,
                )
                probs, _ = predict(model, eval_loader, device)
                pred = split_index[["video_id", "split", "frame", "time_s", f"{target}_label"]].copy()
                pred["target"] = target
                pred["architecture"] = arch
                pred["repeat_seed"] = seed
                pred["risk"] = probs
                pred.to_csv(run_dir / "features" / f"predictions_{target}_seed{seed}_{arch}.csv", index=False)
                for metric in video_metrics(pred, target):
                    metric.update(
                        {
                            "target": target,
                            "architecture": arch,
                            "repeat_seed": seed,
                            "train_time_s": train_time_s,
                            "model_size_bytes": model_size,
                        }
                    )
                    all_metrics.append(metric)
                pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "crop_stability_metrics.csv", index=False)
                pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_stability_training_history.csv", index=False)

    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "crop_stability_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_stability_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "crop_stability_split_counts.csv", index=False)
    summary = summarize(metrics)
    summary.to_csv(run_dir / "metrics" / "crop_stability_summary.csv", index=False)

    lines = ["# Crop CNN Repeated Split Stability", ""]
    lines.append("Parent videos are re-split by attention/blouse label combination. Crops inherit the parent-video split.")
    lines.append("")
    lines.append("| target | architecture | split | AP mean | AP std | F1 mean | balanced acc mean |")
    lines.append("|---|---|---|---:|---:|---:|---:|")
    for _, row in summary.sort_values(["target", "split", "ap_mean"], ascending=[True, True, False]).iterrows():
        lines.append(
            f"| {row['target']} | {row['architecture']} | {row['split']} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | {row['f1_mean']:.3f} | {row['balanced_accuracy_mean']:.3f} |"
        )
    (run_dir / "crop_stability_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Crop CNN Repeated Split Stability", f"- Seeds: `{args.seeds}`\n- Summary: `{run_dir / 'crop_stability_summary.md'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated split validation for crop CNN attention/PPE models.")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--run-name", default="exp_016_crop_cnn_stability")
    parser.add_argument("--architectures", nargs="+", default=["small_cnn", "mobilenet_v3_small", "resnet18"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[111, 222, 333])
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-pretrained", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_016_crop_cnn_stability_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["crop_cnn_stability_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
